<a href="https://colab.research.google.com/github/Attar9132/NYC-taxi-fare-prediction/blob/main/NYC_Taxi_Fare_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**New York Taxi Fare Prediction**


In [ ]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import folium
import random
from geopy.distance import geodesic
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

### Load the dataset from the provided CSV file(TaxiFare.csv).

In [ ]:
# Load the dataset
data = pd.read_csv('TaxiFare.csv')

**Data preprocessing** ensures that data is in a format suitable for modelling, improves model performance, and facilitates accurate taxi fare prediction.

In [ ]:
# Data preprocessing
data['date_time_of_pickup'] = pd.to_datetime(data['date_time_of_pickup'])

In [ ]:
# Handling Missing Values
data.dropna(inplace=True)  # Remove rows with missing values
# Handling Duplicates
data.drop_duplicates(inplace=True)  # Remove duplicate rows

Data Cleaning removes outliers from the dataset by calculating z-scores and identifies data points that fall outside threshold.

In [ ]:
# Calculate the z-scores for each numerical column
z_scores = (data.select_dtypes(include=np.number) - data.select_dtypes(include=np.number).mean()) / data.select_dtypes(include=np.number).std()

# Define a threshold for outliers (e.g., z-score > 3 or z-score < -3)
threshold = 3

# Identify outliers
outliers = np.abs(z_scores) > threshold

# Remove outliers
data = data[~outliers.any(axis=1)]

# Print the updated dataset
print(data)


In [ ]:
selected_features = ['unique_id', 'date_time_of_pickup', 'longitude_of_pickup', 'latitude_of_pickup', 'longitude_of_dropoff', 'latitude_of_dropoff', 'no_of_passenger']
data = data[selected_features + ['amount']]

In [ ]:

# Define the selected features
selected_features = ['unique_id', 'date_time_of_pickup', 'longitude_of_pickup', 'latitude_of_pickup', 'longitude_of_dropoff', 'latitude_of_dropoff', 'no_of_passenger']

# Specify the target variable
target = 'amount'

# Select the features and target variable
data = data[selected_features + [target]]


In [ ]:
data = pd.read_csv('TaxiFare.csv')
# Drop unnecessary columns
data = data.drop(['unique_id'], axis=1)

# Convert date_time_of_pickup column to datetime type
data['date_time_of_pickup'] = pd.to_datetime(data['date_time_of_pickup'])

# Extract additional features from the date_time_of_pickup column
data['hour_of_day'] = data['date_time_of_pickup'].dt.hour
data['day_of_week'] = data['date_time_of_pickup'].dt.dayofweek
data['month'] = data['date_time_of_pickup'].dt.month

# Calculate the distance between pickup and dropoff locations
from geopy.distance import geodesic

def calculate_distance(row):
    pickup_coords = (row['latitude_of_pickup'], row['longitude_of_pickup'])
    dropoff_coords = (row['latitude_of_dropoff'], row['longitude_of_dropoff'])

    if -90 <= pickup_coords[0] <= 90 and -90 <= dropoff_coords[0] <= 90:
        distance = geodesic(pickup_coords, dropoff_coords).miles
    else:
        distance = None

    return distance

data['distance'] = data.apply(calculate_distance, axis=1)

# Drop rows with missing or invalid latitude values
data = data.dropna(subset=['distance'])

# Select relevant features and target variable
selected_features = ['hour_of_day', 'day_of_week', 'month', 'latitude_of_pickup', 'longitude_of_pickup', 'latitude_of_dropoff', 'longitude_of_dropoff', 'no_of_passenger']
target = 'amount'
data = data[selected_features + [target]]

Geopy library implements the geodesic distance formula through its geodesic function.

In [ ]:
# Calculate the distance between pickup and dropoff locations
from geopy.distance import geodesic

def calculate_distance(row):
    pickup_coords = (row['latitude_of_pickup'], row['longitude_of_pickup'])
    dropoff_coords = (row['latitude_of_dropoff'], row['longitude_of_dropoff'])

    if -90 <= pickup_coords[0] <= 90 and -90 <= dropoff_coords[0] <= 90:
        distance = geodesic(pickup_coords, dropoff_coords).miles
    else:
        distance = None

    return distance

In [ ]:
from sklearn.model_selection import train_test_split

X = data[selected_features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)



In [ ]:
y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print("RMSE:", rmse)

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# Make predictions on the test set
y_pred = model.predict(X_test)
# Define the true values (y_test)
y_test = data.loc[X_test.index][target]
# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Print the evaluation metrics
print("Mean Absolute Error (MAE):", mae)
print("R-squared (R2):", r2)

In [ ]:
# Calculate the percentage of predictions within a certain error threshold
threshold = 3
absolute_error = np.abs(y_test - y_pred)
within_threshold = np.sum(absolute_error <= threshold) / len(y_test) * 100
print("Percentage within threshold:", within_threshold)


# **DATA VISUALIZATION**

In [ ]:
import matplotlib.pyplot as plt

# Plot predicted vs. actual fare amount
plt.scatter(y_test, y_pred)
plt.xlabel('Actual Fare Amount')
plt.ylabel('Predicted Fare Amount')
plt.title('Predicted vs. Actual Fare Amount')
plt.show()


In [ ]:
# Calculate residuals
residuals = y_test - y_pred

# Plot residuals
plt.scatter(y_pred, residuals)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted Fare Amount')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.show()


In [ ]:
# Histogram of the fare amount
plt.figure(figsize=(10, 6))
sns.histplot(data['amount'], bins=20, kde=True)
plt.title('Histogram of Fare Amount')
plt.xlabel('Fare Amount')
plt.ylabel('Frequency')
plt.show()

# correlation matrix
corr_matrix = data.corr()

# Heatmap of the correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdYlBu')
plt.title('Correlation Matrix')
plt.show()

# **Prediction of Taxi Fare**

In [ ]:
# Clean the data
data = data[(data['latitude_of_pickup'] >= -90) & (data['latitude_of_pickup'] <= 90)]
data = data[(data['latitude_of_dropoff'] >= -90) & (data['latitude_of_dropoff'] <= 90)]

# Define the latitude and longitude range for random locations (Dataset contains irrelavant lattitude range )
min_lat = -90.0
max_lat = 90.0
min_lon = -180.0
max_lon = 180.0

# To Choose random pickup & dropoff coordinates

pickup_lat = random.uniform(min_lat, max_lat)
pickup_lon = random.uniform(min_lon, max_lon)
dropoff_lat = random.uniform(min_lat, max_lat)
dropoff_lon = random.uniform(min_lon, max_lon)

# Calculate the distance b/w pickup and dropoff locations
pickup_coords = (pickup_lat, pickup_lon)
dropoff_coords = (dropoff_lat, dropoff_lon)
distance = geodesic(pickup_coords, dropoff_coords).miles

# Select appropriate features and target variable
selected_features = ['no_of_passenger']
target = 'amount'
data = data[selected_features + [target]]

# Split the data into training and testing sets
X = data[selected_features]
y = data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the model
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)
# Function to predict fare
def predict_fare_amount(distance, passenger_count):
    # Create a DataFrame with the input features
    data = pd.DataFrame({
        'no_of_passenger': [passenger_count]
    })

    fare_amount = model.predict(data)[0]

    return fare_amount

# For eg
passenger_count = 2

fare_prediction = predict_fare_amount(distance, passenger_count)
print("Predicted Fare Amount:", fare_prediction)


### Visuals on NYC Maps

In [ ]:
import folium
data = pd.read_csv('TaxiFare.csv')
# Create a map centered on NYC
nyc_map = folium.Map(location=[40.7128, -74.0060], zoom_start=10)

# Add markers for pickup locations
for index, row in data.iterrows():
    pickup_location = [row['latitude_of_pickup'], row['longitude_of_pickup']]
    folium.CircleMarker(
        location=pickup_location,
        radius=2,
        color='blue',
        fill=True,
        fill_color='blue'
    ).add_to(nyc_map)

# Display the map
nyc_map
